# Paper Replication: Naive Data Leakage & Transfer Learning

## Overview
This notebook replicates the paper's methodology to demonstrate:
- **Phase 1**: The "Naive" 80/20 Random Split (where leakage happens)
- **Phase 2**: Standard preprocessing (one-hot encoding + Z-standardization)
- **Phase 3**: Cascading prediction strategy (10 1/s → 1 & 100 1/s)
- **Phase 4**: Baseline models on combined dataset (showing high accuracy from replicate leakage)
- **Phase 5**: Transfer Learning experiment (Batch 1 → Batch 2 with covariate shift + LightGBM recovery)

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

# Column definitions
TARGET_1 = "Viscosity_at_shear_rate_1_1/s"
TARGET_10 = "Viscosity_at_shear_rate_10_1/s"
TARGET_100 = "Viscosity_at_shear_rate_100_1/s"
TARGET_COLS = [TARGET_1, TARGET_10, TARGET_100]

CAT_COLS = ["Formulation", "Dispersent_Type"]
CONT_COLS = ["Solid_Content_pct", "Solid_Additive_pct"]
BATCH_COL = "Source_Batch"

# Load combined raw data
DATA_PATH = Path("../data/processed/replication_data.csv")
df = pd.read_csv(DATA_PATH)
print(f"Total rows in combined dataset: {len(df)}")
print(f"Batch composition: {df[BATCH_COL].value_counts().to_dict()}")

print(df.head())

Total rows in combined dataset: 178
Batch composition: {'Batch_1': 91, 'Batch_2': 87}
  Formulation Dispersent_Type  Solid_Content_pct  Solid_Additive_pct  \
0          F1        Hypermer               73.0                0.50   
1          F1        Hypermer               73.0                0.25   
2          F1        Hypermer               73.0                0.25   
3          F1        Hypermer               77.0                0.25   
4          F1        Hypermer               73.0                0.25   

   Viscosity_at_shear_rate_1_1/s  Viscosity_at_shear_rate_10_1/s  \
0                       10.56640                         3.78921   
1                       71.65190                        14.08460   
2                        9.64639                         3.26827   
3                       61.11070                        18.77450   
4                        8.37111                         4.83186   

   Viscosity_at_shear_rate_100_1/s Source_Batch  
0                     

# Data Pre-processing

- Train Test Split
- One-hot encode categorical columns
- Z-standardise continuous columns

In [3]:
# Randomly split into train/test sets 
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Train rows: {len(df_train)} ({len(df_train)/len(df):.1%})")
print(f"Test rows: {len(df_test)} ({len(df_test)/len(df):.1%})")

# Features/targets split
X_train = df_train[CAT_COLS + CONT_COLS].copy()
X_test = df_test[CAT_COLS + CONT_COLS].copy()
y_train = df_train[TARGET_COLS].copy()
y_test = df_test[TARGET_COLS].copy()

# 1) One-hot encode categorical columns
X_train = pd.get_dummies(X_train, columns=CAT_COLS, drop_first=False)
X_test = pd.get_dummies(X_test, columns=CAT_COLS, drop_first=False)

# Align test columns to training columns (prevents leakage from test-only categories)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# 2) Z-standardize continuous columns (fit on train only, transform both)
scaler = StandardScaler()
X_train[CONT_COLS] = scaler.fit_transform(X_train[CONT_COLS])
X_test[CONT_COLS] = scaler.transform(X_test[CONT_COLS])

print("X_train shape:", X_train.shape, "| X_test shape:", X_test.shape)
print("Aligned columns:", X_train.columns.equals(X_test.columns))

Train rows: 142 (79.8%)
Test rows: 36 (20.2%)
X_train shape: (142, 10) | X_test shape: (36, 10)
Aligned columns: True


In [4]:
# Paper-appendix XGBoost baseline for 10 1/s
xgb_10 = XGBRegressor(
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    n_estimators=100,
    objective="reg:squarederror",
    random_state=RANDOM_STATE
)

# Train only for the intermediate target (10 1/s)
xgb_10.fit(X_train, y_train[TARGET_10])

# Predict on the naive 20% test split
y10_pred = xgb_10.predict(X_test)
y10_true = y_test[TARGET_10].values

# Baseline metrics
r2_10 = r2_score(y10_true, y10_pred)
rmse_10 = np.sqrt(mean_squared_error(y10_true, y10_pred))
mae_10 = mean_absolute_error(y10_true, y10_pred)

print("XGBoost Baseline (10 1/s) - Naive 80/20 Split")
print(f"R²:   {r2_10:.4f}")
print(f"RMSE: {rmse_10:.4f}")
print(f"MAE:  {mae_10:.4f}")

XGBoost Baseline (10 1/s) - Naive 80/20 Split
R²:   0.8834
RMSE: 3.2600
MAE:  2.1416


In [5]:
# --- Cascading Step: inject predicted 10 1/s as an additional feature ---
train_pred_10 = xgb_10.predict(X_train)
test_pred_10 = xgb_10.predict(X_test)

X_train_cascade = X_train.copy()
X_test_cascade = X_test.copy()
X_train_cascade["Predicted_Visc_10"] = train_pred_10
X_test_cascade["Predicted_Visc_10"] = test_pred_10

print("Cascade feature added:", "Predicted_Visc_10")
print("X_train_cascade shape:", X_train_cascade.shape, "| X_test_cascade shape:", X_test_cascade.shape)

# Fresh XGBoost model for 1 1/s target (same paper hyperparameters)
xgb_1 = XGBRegressor(
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    n_estimators=100,
    objective="reg:squarederror",
    random_state=RANDOM_STATE
)
xgb_1.fit(X_train_cascade, y_train[TARGET_1])

# Fresh XGBoost model for 100 1/s target (same paper hyperparameters)
xgb_100 = XGBRegressor(
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    n_estimators=100,
    objective="reg:squarederror",
    random_state=RANDOM_STATE
)
xgb_100.fit(X_train_cascade, y_train[TARGET_100])

# Predict extremes on test set
y1_pred = xgb_1.predict(X_test_cascade)
y100_pred = xgb_100.predict(X_test_cascade)

# Metrics for 1 1/s
r2_1 = r2_score(y_test[TARGET_1], y1_pred)
rmse_1 = np.sqrt(mean_squared_error(y_test[TARGET_1], y1_pred))
mae_1 = mean_absolute_error(y_test[TARGET_1], y1_pred)

# Metrics for 100 1/s
r2_100 = r2_score(y_test[TARGET_100], y100_pred)
rmse_100 = np.sqrt(mean_squared_error(y_test[TARGET_100], y100_pred))
mae_100 = mean_absolute_error(y_test[TARGET_100], y100_pred)

print("\nXGBoost Cascading Results (Naive 80/20 Split)")
print("Target 1 1/s")
print(f"R²:   {r2_1:.4f}")
print(f"RMSE: {rmse_1:.4f}")
print(f"MAE:  {mae_1:.4f}")

print("\nTarget 100 1/s")
print(f"R²:   {r2_100:.4f}")
print(f"RMSE: {rmse_100:.4f}")
print(f"MAE:  {mae_100:.4f}")

Cascade feature added: Predicted_Visc_10
X_train_cascade shape: (142, 11) | X_test_cascade shape: (36, 11)

XGBoost Cascading Results (Naive 80/20 Split)
Target 1 1/s
R²:   0.6269
RMSE: 27.8583
MAE:  16.1812

Target 100 1/s
R²:   0.8825
RMSE: 1.0172
MAE:  0.6197


# Phase 5: Transfer Learning — Batch 1 → Batch 2 (LightGBM Update)

Replicate the paper's core TL finding by simulating a factory batch shift:

- **5.1** Separate each batch; fit the scaler *only* on Batch 1, apply to both.
- **5.2** Split Batch 2 into 20% fine-tuning / 80% holdout.
- **5.3** Train a 500-tree LightGBM source model on Batch 1 (`lr=0.10`).
- **5.4** **No-TL baseline** — test the Batch 1 model directly on the holdout (expect low/negative R²).
- **5.5** **Transfer** — pass the 500-tree model into `init_model` and append 100 new trees (`lr=0.05`) fine-tuned on the 20% slice.
- **5.6** Final evaluation on the holdout → R² jump confirms TL recovers batch shift.

In [6]:
import lightgbm as lgb

# 5.1  Separate batches and preprocess (scaler fitted on Batch 1 only)
df_b1 = df[df[BATCH_COL] == "Batch_1"].copy().reset_index(drop=True)
df_b2 = df[df[BATCH_COL] == "Batch_2"].copy().reset_index(drop=True)

X_b1 = pd.get_dummies(df_b1[CAT_COLS + CONT_COLS], columns=CAT_COLS, drop_first=False)
X_b2 = pd.get_dummies(df_b2[CAT_COLS + CONT_COLS], columns=CAT_COLS, drop_first=False)

# Align Batch 2 columns to Batch 1 schema
X_b1, X_b2 = X_b1.align(X_b2, join="left", axis=1, fill_value=0)

scaler_tl = StandardScaler()
X_b1[CONT_COLS] = scaler_tl.fit_transform(X_b1[CONT_COLS])   # fit on source only
X_b2[CONT_COLS] = scaler_tl.transform(X_b2[CONT_COLS])       # apply to target

y_b1 = df_b1[TARGET_COLS]
y_b2 = df_b2[TARGET_COLS].reset_index(drop=True)

print(f"Batch 1: {len(X_b1)} rows  |  Batch 2: {len(X_b2)} rows  |  Features: {X_b1.shape[1]}")

# 5.2  Split Batch 2 -> 20% fine-tune / 80% holdout
X_b2_ft, X_b2_hold, y_b2_ft, y_b2_hold = train_test_split(
    X_b2, y_b2, test_size=0.80, random_state=RANDOM_STATE
)
print(f"Batch 2  fine-tune: {len(X_b2_ft)} rows  |  holdout: {len(X_b2_hold)} rows")

# 5.3  Source model: 500-tree LightGBM on Batch 1 via native API (lr=0.10)
src_params = {
    "learning_rate": 0.10,
    "verbose": -1,
    "seed": RANDOM_STATE,
    "min_data_in_leaf": 5,
}
src_data = lgb.Dataset(X_b1, label=y_b1[TARGET_10].values)
lgbm_src_booster = lgb.train(src_params, src_data, num_boost_round=500)
print(f"\nSource model trained -> {lgbm_src_booster.num_trees()} trees")

# 5.4  No-TL baseline: source model tested directly on Batch 2 holdout
y10_no_tl  = lgbm_src_booster.predict(X_b2_hold)
r2_no_tl   = r2_score(y_b2_hold[TARGET_10], y10_no_tl)
rmse_no_tl = np.sqrt(mean_squared_error(y_b2_hold[TARGET_10], y10_no_tl))
mae_no_tl  = mean_absolute_error(y_b2_hold[TARGET_10], y10_no_tl)

print("\n-- No-TL Baseline (Batch 1 model -> Batch 2 holdout) ----------")
print(f"   R2:   {r2_no_tl:.4f}")
print(f"   RMSE: {rmse_no_tl:.4f}")
print(f"   MAE:  {mae_no_tl:.4f}")

# 5.5  Transfer: continue training from the source booster.
#
#      In LightGBM >= 4.x, num_boost_round is ADDITIONAL rounds when init_model
#      is provided. Set to 100 to append exactly 100 new trees -> 600 total.
#
#      min_data_in_leaf=1 is required so that LightGBM can split the small
#      17-row fine-tune slice (the default of 20 would block all splits).
tl_params = {
    "learning_rate": 0.05,
    "verbose": -1,
    "seed": RANDOM_STATE,
    "min_data_in_leaf": 1,
}
ft_data = lgb.Dataset(X_b2_ft, label=y_b2_ft[TARGET_10].values)
lgbm_tl_booster = lgb.train(
    tl_params,
    ft_data,
    num_boost_round=100,            # 100 additional trees appended to the 500 source trees
    init_model=lgbm_src_booster
)
print(f"\nUpdated model -> {lgbm_tl_booster.num_trees()} trees (500 frozen + 100 appended)")

# 5.6  Final TL evaluation on the 80% Batch 2 holdout
y10_tl   = lgbm_tl_booster.predict(X_b2_hold)
r2_tl    = r2_score(y_b2_hold[TARGET_10], y10_tl)
rmse_tl  = np.sqrt(mean_squared_error(y_b2_hold[TARGET_10], y10_tl))
mae_tl   = mean_absolute_error(y_b2_hold[TARGET_10], y10_tl)

print("\n-- TL Model (600 trees, fine-tuned on Batch 2 20%) -> Batch 2 holdout --")
print(f"   R2:   {r2_tl:.4f}")
print(f"   RMSE: {rmse_tl:.4f}")
print(f"   MAE:  {mae_tl:.4f}")
print(f"\n   R2 improvement from TL: {r2_tl - r2_no_tl:+.4f}")

Batch 1: 91 rows  |  Batch 2: 87 rows  |  Features: 9
Batch 2  fine-tune: 17 rows  |  holdout: 70 rows

Source model trained -> 500 trees

-- No-TL Baseline (Batch 1 model -> Batch 2 holdout) ----------
   R2:   0.2638
   RMSE: 10.1813
   MAE:  7.0544

Updated model -> 600 trees (500 frozen + 100 appended)

-- TL Model (600 trees, fine-tuned on Batch 2 20%) -> Batch 2 holdout --
   R2:   0.5750
   RMSE: 7.7355
   MAE:  5.5230

   R2 improvement from TL: +0.3112


In [ ]:
import lightgbm as lgb

# 5.1  Separate batches and preprocess (scaler fitted on Batch 1 only)
df_b1 = df[df[BATCH_COL] == "Batch_1"].copy().reset_index(drop=True)
df_b2 = df[df[BATCH_COL] == "Batch_2"].copy().reset_index(drop=True)

X_b1 = pd.get_dummies(df_b1[CAT_COLS + CONT_COLS], columns=CAT_COLS, drop_first=False)
X_b2 = pd.get_dummies(df_b2[CAT_COLS + CONT_COLS], columns=CAT_COLS, drop_first=False)

# Align Batch 2 columns to Batch 1 schema
X_b1, X_b2 = X_b1.align(X_b2, join="left", axis=1, fill_value=0)

scaler_tl = StandardScaler()
X_b1[CONT_COLS] = scaler_tl.fit_transform(X_b1[CONT_COLS])   # fit on source only
X_b2[CONT_COLS] = scaler_tl.transform(X_b2[CONT_COLS])       # apply to target

y_b1 = df_b1[TARGET_COLS]
y_b2 = df_b2[TARGET_COLS].reset_index(drop=True)

print(f"Batch 1: {len(X_b1)} rows  |  Batch 2: {len(X_b2)} rows  |  Features: {X_b1.shape[1]}")

# 5.2  Split Batch 2 -> 20% fine-tune / 80% holdout
X_b2_ft, X_b2_hold, y_b2_ft, y_b2_hold = train_test_split(
    X_b2, y_b2, test_size=0.80, random_state=RANDOM_STATE
)
print(f"Batch 2  fine-tune: {len(X_b2_ft)} rows  |  holdout: {len(X_b2_hold)} rows")

# 5.3  Source model: 500-tree LightGBM on Batch 1 via native API (lr=0.10)
src_params = {
    "learning_rate": 0.10,
    "verbose": -1,
    "seed": RANDOM_STATE,
    "min_data_in_leaf": 5,
}
src_data = lgb.Dataset(X_b1, label=y_b1[TARGET_10].values)
lgbm_src_booster = lgb.train(src_params, src_data, num_boost_round=500)
print(f"\nSource model trained -> {lgbm_src_booster.num_trees()} trees")

# 5.4  No-TL baseline: source model tested directly on Batch 2 holdout
y10_no_tl  = lgbm_src_booster.predict(X_b2_hold)
r2_no_tl   = r2_score(y_b2_hold[TARGET_10], y10_no_tl)
rmse_no_tl = np.sqrt(mean_squared_error(y_b2_hold[TARGET_10], y10_no_tl))
mae_no_tl  = mean_absolute_error(y_b2_hold[TARGET_10], y10_no_tl)

print("\n-- No-TL Baseline (Batch 1 model -> Batch 2 holdout) ----------")
print(f"   R2:   {r2_no_tl:.4f}")
print(f"   RMSE: {rmse_no_tl:.4f}")
print(f"   MAE:  {mae_no_tl:.4f}")

# 5.5  Transfer: continue training from the source booster.
#
#      In LightGBM >= 4.x, num_boost_round is ADDITIONAL rounds when init_model
#      is provided. Set to 100 to append exactly 100 new trees -> 600 total.
#
#      min_data_in_leaf=1 is required so that LightGBM can split the small
#      17-row fine-tune slice (the default of 20 would block all splits).
tl_params = {
    "learning_rate": 0.05,
    "verbose": -1,
    "seed": RANDOM_STATE,
    "min_data_in_leaf": 1,
}
ft_data = lgb.Dataset(X_b2_ft, label=y_b2_ft[TARGET_10].values)
lgbm_tl_booster = lgb.train(
    tl_params,
    ft_data,
    num_boost_round=100,            # 100 additional trees appended to the 500 source trees
    init_model=lgbm_src_booster
)
print(f"\nUpdated model -> {lgbm_tl_booster.num_trees()} trees (500 frozen + 100 appended)")

# 5.6  Final TL evaluation on the 80% Batch 2 holdout
y10_tl   = lgbm_tl_booster.predict(X_b2_hold)
r2_tl    = r2_score(y_b2_hold[TARGET_10], y10_tl)
rmse_tl  = np.sqrt(mean_squared_error(y_b2_hold[TARGET_10], y10_tl))
mae_tl   = mean_absolute_error(y_b2_hold[TARGET_10], y10_tl)

print("\n-- TL Model (600 trees, fine-tuned on Batch 2 20%) -> Batch 2 holdout --")
print(f"   R2:   {r2_tl:.4f}")
print(f"   RMSE: {rmse_tl:.4f}")
print(f"   MAE:  {mae_tl:.4f}")
print(f"\n   R2 improvement from TL: {r2_tl - r2_no_tl:+.4f}")

Batch 1: 87 rows  |  Batch 2: 91 rows  |  Features: 10
Batch 2  fine-tune: 18 rows  |  holdout: 73 rows

Source model trained -> 500 trees

-- No-TL Baseline (Batch 1 model -> Batch 2 holdout) ----------
   R2:   0.8092
   RMSE: 5.3692
   MAE:  3.5889

Updated model -> 600 trees (500 frozen + 100 appended)

-- TL Model (600 trees, fine-tuned on Batch 2 20%) -> Batch 2 holdout --
   R2:   0.8002
   RMSE: 5.4936
   MAE:  3.8558

   R2 improvement from TL: -0.0089
